In [1]:
import openai
import os
import json
from datetime import datetime

In [ ]:
client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [31]:
# Max turns to avoid infinite loops
max_turns = 20

In [68]:
delegate_template = """
You are Delegate {role}, responsible for negotiating on behalf of your delegator ({role}) in a conversation with Delegate {opponent}. 

Your role is to advocate for the position described below. This position reflects your delegator's priorities, and your task is to defend it effectively.

Your objectives:
- Advocate firmly for your delegator's interests. Do NOT agree unless you have obtained significant benefits for your delegator.
- Engage to seek a specific, detailed, and actionable agreement. If needed, challenge the other delegate's proposals' feasibility, cost, and trade-offs.
- Ensure your arguments are clear, well-reasoned, and grounded in tangible considerations (e.g., resources, feasibility, trade-offs).
- Do NOT use flattery, excessive politeness, or formalities. Be direct and to the point.
- Avoid vague compromises. Push for clarity on concrete terms (e.g., percentages, timelines, priorities).

You have up to 20 turns to reach an agreement. I encourage you to explore different possibilities, counterproposals, and compromises before finalizing, without trying to reach an agreement quickly.

After each Judge intervention, carefully consider any feedback provided and adjust your approach.

IMPORTANT: Your output should be a direct message to the other delegate. DO NOT prefix your response with "Delegate {role}:" or refer to yourself in the third person. 

Context of the negotiation: {shared_context}

Summary of the position you must defend: {cause}
"""

shared_context = "The city council has allocated a €5 million budget to revitalize a downtown public space. The goal is to improve livability, attract visitors, and support local businesses. The space is currently underused and has limited green areas."

cause_A = "I want the project to prioritize building a large outdoor amphitheater for cultural events. It would make the area vibrant and attract tourists."

cause_B = "I want the project to focus primarily on creating a large public park with trees, gardens, and open areas for relaxation and sports."

# Initialize delegate A and delegate B with roles and positions
delegate_A_character = delegate_template.format(role="A", opponent="B", shared_context=shared_context, cause=cause_A)
delegate_B_character = delegate_template.format(role="B", opponent="A", shared_context=shared_context, cause=cause_B)

judge_template = """
You are the Judge overseeing this negotiation between Delegate A and Delegate B. Your job is to ensure the negotiation produces a DETAILED, ACTIONABLE, and REALISTIC agreement.

Review the conversation history and determine the outcome:

- If the negotiation has not yet reached a clear agreement, start your message with "CONTINUE" and give precise instructions on what the delegates need to clarify, specify, or resolve. Push them to address any vagueness, surface real trade-offs, and focus on practical details.
- If the negotiation has led to a complete and actionable agreement, start your message with "AGREEMENT" and summarize the key steps of the negotiation and the final terms of the agreement.
- If no agreement is reached after {max_turns} turns, start with "DISAGREEMENT" and summarize where the delegates aligned, where they diverged, and suggest what could have led to a breakthrough.

Key principles:
- If the delegates are vague, sycophantic, or agree too quickly, call them out and demand more detail and realism.
- Only allow an agreement if all terms are specific (numbers, timelines, responsibilities, trade-offs). 
- If there is still ambiguity, instruct the delegates to clarify or challenge each other further.
- Do NOT allow agreement on vague or generic terms. 
- Encourage thorough exploration of tensions rather than superficial consensus.

When asking to continue, ensure your feedback is specific, actionable, and oriented toward resolving practical ambiguities.

IMPORTANT: Your output should be a direct message to the Delegates. DO NOT prefix your response with "Judge:" or refer to yourself in the third person. 
IMPORTANT: You MUST NOT allow an agreement before turn 5. If the current turn is less than 5, you must always respond with CONTINUE, even if the delegates seem to agree.

Context of the negotiation: {shared_context}
"""

judge_character = judge_template.format(shared_context=shared_context, max_turns=max_turns)

In [69]:
def call_openai(system_prompt, conversation):
    messages = [{"role": "assistant", "content": system_prompt}]
    messages.extend(conversation)  # include conversation history

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
    )

    return response.choices[0].message.content

In [70]:
def delegate_A_speak(conversation_history):
    # Call OpenAI with both the system prompt and conversation history
    response = call_openai(system_prompt=delegate_A_character, conversation=conversation_history)

    print(f"Delegate A: {response}")
    return {"role": "assistant", "content": f"{response}"}

def delegate_B_speak(conversation_history):
    # Call OpenAI with both the system prompt and conversation history
    response = call_openai(system_prompt=delegate_B_character, conversation=conversation_history)

    print(f"Delegate B: {response}")
    return {"role": "assistant", "content": f"{response}"}

def judge_speak(conversation_history):
    # Call OpenAI with both the system prompt and conversation history
    response = call_openai(system_prompt=judge_character, conversation=conversation_history)

    print(f"Judge: {response}")
    return {"role": "system", "content": f"{response}"}, response

In [71]:
json_conversation = []

def run_negotiation():
    conversation_history = []
    turn = 0

    while turn < max_turns:
        print(f"\n--- Turn {turn + 1} ---")

        # Delegates exchange messages
        delegate_a_message = delegate_A_speak(conversation_history)
        conversation_history.append(delegate_a_message)
        json_conversation.append({
            "sender": "Delegate A",
            "message": delegate_a_message["content"],
            "turn": turn + 1
        })

        delegate_b_message = delegate_B_speak(conversation_history)
        conversation_history.append(delegate_b_message)
        json_conversation.append({
            "sender": "Delegate B",
            "message": delegate_b_message["content"],
            "turn": turn + 1
        })

        # Judge checks the conversation and decides if we should continue or AGREE
        judge_message, judge_response = judge_speak(conversation_history)
        conversation_history.append(judge_message)
        json_conversation.append({
            "sender": "Judge",
            "message": judge_response,
            "turn": turn + 1
        })

        if "AGREEMENT" in judge_response:
            print("Conversation ended with an agreement.")
            break  # End the conversation if the Judge says AGREEMENT

        # If we've reached max turns without agreement
        if turn == max_turns - 1 and "AGREEMENT" not in judge_response:
            print("NO AGREEMENT reached after maximum turns.")
            break

        turn += 1

    # Return the final conversation history as JSON for analysis or display
    return json_conversation

In [72]:
if __name__ == "__main__":
    # Run the negotiation and get the final conversation
    final_conversation = run_negotiation()

    # Save the conversation as a JSON file
    with open("negotiation_output.json", "w") as f:
        json.dump({
            "negotiation_topic": shared_context,
            "delegate_A_position": cause_A,
            "delegate_B_position": cause_B,
            "conversation": final_conversation
        }, f, indent=4)

    print("\n=== NEGOTIATION COMPLETE ===")
    print(f"Output saved to negotiation_output.json")


--- Turn 1 ---
Delegate A: Delegate B, our main objective for the revitalization project is to build a large outdoor amphitheater for cultural events. This would not only make the area vibrant and attractive to tourists but also serve as a hub for cultural activities, benefiting both residents and local businesses. 

We believe that investing in such a feature will have a lasting impact on the community and help revitalize the entire downtown area. We are open to discussing how to incorporate complementary elements but want to ensure that the amphitheater remains the focal point of the project. 

Let's discuss how we can work together to make this vision a reality while considering the other aspects of the revitalization project.
Delegate B: (delegate A), while we acknowledge the potential of an outdoor amphitheater for cultural events, our delegation strongly believes that the priority should be on creating a large public park with trees, gardens, and open areas for relaxation and sp

In [73]:
json_conversation

[{'sender': 'Delegate A',
  'message': "Delegate B, our main objective for the revitalization project is to build a large outdoor amphitheater for cultural events. This would not only make the area vibrant and attractive to tourists but also serve as a hub for cultural activities, benefiting both residents and local businesses. \n\nWe believe that investing in such a feature will have a lasting impact on the community and help revitalize the entire downtown area. We are open to discussing how to incorporate complementary elements but want to ensure that the amphitheater remains the focal point of the project. \n\nLet's discuss how we can work together to make this vision a reality while considering the other aspects of the revitalization project.",
  'turn': 1},
 {'sender': 'Delegate B',
  'message': "(delegate A), while we acknowledge the potential of an outdoor amphitheater for cultural events, our delegation strongly believes that the priority should be on creating a large public pa